In [2]:
import pydough

#%load_ext pydough.jupyter_extensions
%reload_ext pydough.jupyter_extensions

#Necessary for comparison
import pandas as pd
from pandas.testing import assert_frame_equal, assert_series_equal
import re
import eval
import datetime

import collections
import numpy as np
import sqlite3 as sql
import os

In [3]:
#YOUR .SQL FILE TO CREATE THE DATABASE, COPY IT TO THIS FOLDER.
SQL_path = 'databases/Defog/init_defog.sql'

#METADATA FOR THE GRAPH .JSON
metadata_path = "metadata/Defog/Ewallet_graph.json"

#GRAPH NAME
graph_name = "Ewallet"

#DESIRED DATABASE NAME
DB_name = "notebookTest.db"



with open(SQL_path, 'r') as sql_file:
    sql_script = sql_file.read()

os.remove(DB_name)
connection = sql.connect(DB_name)
cursor = connection.cursor()
cursor.executescript(sql_script)

pydough.active_session.load_metadata_graph(metadata_path, graph_name)
pydough.active_session.connect_database("sqlite", database=DB_name)

DatabaseContext(connection=<pydough.database_connectors.database_connector.DatabaseConnection object at 0x7bf0fa4fbfe0>, dialect=<DatabaseDialect.SQLITE: 'sqlite'>)

In [3]:
tested_file, tested_df = eval.compare_output("evalNotebookFiles/", "evalNotebookFiles/noMatch.csv", ".", ".")

DATEDIFF unsupported for 'DAYS'.


How many sales did each salesperson make in the past 30 days, inclusive of today's date? Return their ID, first name, last name and number of sales made, ordered from most to least sales. Dealership
Return each doctor's doc_id, specialty, number of distinct drugs prescribed, and SDRSDR = a doctor's rank within their specialty by number of distinct drugs prescribed. Doctors prescribing more drugs will have a higher rank DermTreatment
Calculate the CPUR for each merchant, considering only successful transactions. Return the merchant name and CPUR.CPUR (coupon usage rate) = number of distinct coupons used / number of distinct transactions Ewallet
What are the top 3 most frequently used coupon codes? Return the coupon code, total number of redemptions, and total amount redeemed. Ewallet


/home/j/miniconda3/envs/aisuite_deepseek/lib/python3.12/site-packages/pydough/sqlglot/sqlglot_relational_expression_visitor.py:103: UserWarning: PyDough when using SQLITE dialect does not support ascending ordering with nulls first (changed to nulls last)
  warnings.warn(


In [21]:
query = '''
SELECT d.drug_name, COUNT(*) AS num_treatments, AVG(t.tot_drug_amt) AS avg_drug_amt FROM treatments AS t JOIN drugs AS d ON t.drug_id = d.drug_id GROUP BY d.drug_name ORDER BY CASE WHEN num_treatments IS NULL THEN 1 ELSE 0 END DESC, num_treatments DESC, CASE WHEN avg_drug_amt IS NULL THEN 1 ELSE 0 END DESC, avg_drug_amt DESC, d.drug_name DESC;'''

sql_output = pd.read_sql_query(query, connection)
sql_output

,drug_name,num_treatments,avg_drug_amt
0,Drugalin,6,206.666667
1,Medicol,6,178.333333
2,Topizol,4,307.500000
3,Biologic-X,4,150.000000
4,Topicort,3,580.000000
5,Smallazine,3,203.333333


In [8]:
%%pydough

user_session_duration = Users.CALCULATE( user_id=uid, total_duration=SUM( sessions.WHERE( (session_start_ts >= '2023-06-01') & (session_end_ts < '2023-06-08') ).CALCULATE( duration_in_seconds=DATEDIFF("seconds", session_start_ts, session_end_ts) ).duration_in_seconds ) ).ORDER_BY(total_duration.DESC())

result_df = pydough.to_df(user_session_duration)
result_sql = pydough.to_sql(user_session_duration)
print(result_df)

DATEDIFF unsupported for 'DAYS'.
DATEDIFF unsupported for 'DAYS'.


    user_id  total_duration
0         6            4205
1         2            3034
2         4            2736
3         8            2735
4         1            2715
5         1            2113
6         5            2098
7        10            1843
8         9            1797
9         7            1518
10        8             905
11        3             748
12        3             622
13        1             190
14        2             190
15       11               0


In [18]:
eval.compare_df(sql_output, result_df, "test", "test")

Info: Not enough rows in df_gen to match all of df_gold's rows: 6 vs 5.
Info: Not enough rows in df_gen to match all of df_gold's rows: 6 vs 5.


False